# ML-02: Research Question and Provisional Lane

Week 1 framing. Data is the anonymized starter slice (`data/raw/content_refresh_anonymized.csv`). The lane below is provisional until the end of week 4, but I'd rather commit to something specific now and change it with evidence than stay vague.

## 1. My lane (or freestyle) and why

Provisional lane: Lane 2, Refresh / Content Opportunity Scoring.

What stuck with me after notebooks 01 and 02 is that this is a capacity problem before it's a prediction problem. More than half the starter pages carry the "down" tag, and no content team is going to open thirteen thousand pages in a month. They might open fifty. So the thing worth building isn't a decline detector. It's an ordering: which pages should a person look at first, and what reason can I give them for each one.

I looked at the other lanes before settling. CTR / Engagement scoring is a good lane, but low CTR on a visible page already fits inside a refresh queue as a reason code, while the opposite isn't true. Clustering would give me nice archetype profiles, but a cluster doesn't tell an editor which page to open on Monday. Ranking Signal Analysis is the most honest of the four, and I'll lean on it anyway during the week 4 signal audit.

The one direction I'm keeping open is the Growth / Recovery freestyle track. The starter label is weak (section 4 shows exactly why), and if the warehouse daily table lets me build a clean "prior 90 days, next 30 days" decline label, the lane might drift that way. The output would still be a ranked review queue.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Walk up from the notebook until the repo's data folder shows up, so this runs
# from work/notebooks/, from the repo root, or from a fresh Colab clone.
def find_repo_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return folder
    raise FileNotFoundError("Run this from inside a clone of the repo.")

ROOT = find_repo_root(Path.cwd().resolve())
df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")

print(f"rows: {len(df):,}   columns: {df.shape[1]}   clients: {df['client_id'].nunique()}")
print(f"one row per page: {df['content_id'].is_unique}")

rows: 30,000   columns: 44   clients: 32
one row per page: True


## 2. The question: decision, action, cost of a wrong call

**Research question.** Among existing pages that still get real search demand, which ones should a content editor review first this cycle, and what should they do once they open each one?

**Unit of analysis.** One content item (a page) at a fixed decision point. In the starter slice that's simply one row per page with trailing 90-day metrics. On the warehouse it becomes page × decision date, which matters a lot once I build time windows, because the same page can be a good candidate in March and a bad one in May.

**Output.** A ranked review queue. The top K pages (K set by review capacity, 50 to start) each get a score, a suggested action (refresh, expand, review title/meta, review on-page engagement, or monitor), the reason codes behind the score, and a confidence label.

**Who acts, and how.** A content strategist or editor working a client account goes down the list. For each page they read the reason codes, check the page, and either schedule a refresh, rewrite the title and meta, expand the section that's thin, or mark it fine and move on. The queue doesn't edit anything. It decides where human time goes.

**Cost of a wrong call.** There are two kinds of mistake and they don't cost the same:

- A false positive (ranked high, didn't need work) burns review time. A quick check is minutes; a rewrite is one to three hours. There's also a small risk that rewriting a healthy page makes it worse.
- A false negative (really slipping, ranked low) costs whatever clicks the page keeps losing until someone notices. That cost grows with demand, so missing a decline on a page with 20,000 impressions is far worse than missing one on a page with 120.

Because of that asymmetry I'll judge the queue at the capacity cutoff (precision@50, precision@20 for a single reviewer) and add a demand-weighted view next to it. Accuracy over all 30,000 rows would reward getting row 25,000 right, and nobody ever reads row 25,000.

**Why data or ML helps at all.** A plain rule like "declining and visible" fires on most of the inventory (the cell below counts it), so it filters without ranking. The signals that might separate real decay from noise, things like position, age, how many days a page actually shows up, engagement, and what the rest of the client's site is doing, interact in ways I can't write cleanly as if-statements. The starter pipeline points the same way: under client holdout the rule baseline got about 12 of its top 50 right and the random forest about 37. That's a reason to try a model. It isn't proof that one works on a label I trust.

**Why this isn't just "train a model".** The hard part here is the label, not the classifier. The starter target `trend_direction == "down"` is computed from the same 30-day windows that describe the page, so it records a drop that already happened instead of predicting one. Weeks 2 and 3 are about defining decline in a later window and separating it from client-wide drops, seasonality, consolidation and noise. Get that wrong and a great precision@50 just means the model learned the rule I gave it.

In [2]:
CAPACITY = 50  # pages one team can realistically review per cycle

down = df["trend_direction"].eq("down")
has_demand = df["impressions_90d"].ge(100)
rule_hits = df[down & has_demand]

print(f"pages tagged 'down' with >= 100 impressions in 90d: {len(rule_hits):,}")
print(f"that is {len(rule_hits) / CAPACITY:,.0f}x a {CAPACITY}-page review queue")
print(f"share of all pages the rule flags: {len(rule_hits) / len(df):.1%}")

pages tagged 'down' with >= 100 impressions in 90d: 13,152
that is 263x a 50-page review queue
share of all pages the rule flags: 43.8%


The simple rule flags 13,152 pages, roughly 263 queues' worth. A filter that keeps 44% of the inventory doesn't make a decision for anyone, which is the whole case for ranking.

## 3. Quick look at the data (2-3 real numbers)

Three numbers decided it for me: how much demand sits on declining pages, how lopsided that demand is, and how much of the "decline" is really a client-level effect.

In [3]:
# 1) How much search demand is sitting on pages that dropped?
d = df[down]
lost = (d["impressions_prev_30d"] - d["impressions_last_30d"]).sum()
before = d["impressions_prev_30d"].sum()
print(f"[1] declining pages: {down.sum():,} ({down.mean():.1%} of all pages)")
print(f"    their impressions fell from {before:,.0f} to {before - lost:,.0f} "
      f"month over month, a {lost / before:.1%} drop")

# 2) Is demand spread evenly, or concentrated in a few pages?
imps = df["impressions_90d"].sort_values(ascending=False)
top10 = imps.head(len(imps) // 10).sum() / imps.sum()
print(f"[2] the top 10% of pages hold {top10:.1%} of all 90-day impressions")

# 3) Does 'decline' behave like a page property or a client property?
client_rate = df.groupby("client_id")["trend_direction"].apply(lambda s: s.eq("down").mean())
client_size = df["client_id"].value_counts()
print(f"[3] share of a client's pages tagged 'down': "
      f"min {client_rate.min():.1%}, median {client_rate.median():.1%}, max {client_rate.max():.1%}")
print(f"    clients with >= 80% of pages down: {(client_rate >= 0.8).sum()} of {len(client_rate)}")
print(f"    largest client holds {client_size.max() / len(df):.1%} of all rows "
      f"(smallest has {client_size.min()} pages)")

[1] declining pages: 16,262 (54.2% of all pages)
    their impressions fell from 29,408,488 to 15,311,594 month over month, a 47.9% drop
[2] the top 10% of pages hold 70.2% of all 90-day impressions
[3] share of a client's pages tagged 'down': min 0.0%, median 52.4%, max 93.7%
    clients with >= 80% of pages down: 2 of 32
    largest client holds 23.4% of all rows (smallest has 3 pages)


What these say, in order.

The pages tagged as declining lost about 14.1M of their 29.4M prior-month impressions, close to half. There's real demand on the table, so a better ordering has something to win.

Demand is very uneven. The top tenth of pages carries about 70% of all impressions. That's why the evaluation needs a demand-weighted view: a queue can score well on precision@50 by finding lots of small declines and still miss the few pages that matter.

The third number changed how I think about the label. The share of a client's pages tagged "down" runs from 0% for one client to almost 94% for another. On the worst client, about 980 of its 1,043 pages dropped in the same month. That looks like a tracking change, seasonality or a site-wide event far more than a thousand separate content problems. Client size is lopsided too, with one client holding about 23% of the rows. So client-holdout validation isn't optional, and I'll probably need features that compare a page to its own client's trend rather than to the global average.

## 4. Careful words: what I can and can't claim

**What this work will be able to say.**

- Observed: statements about the slice as measured, such as "in this starter slice, 54% of pages had last-30-day impressions more than 20% below the 30 days before."
- Directional: "pages with X tended to show Y more often," with the size of the gap and the sample it came from.
- Decision-support: "on clients the model never saw, this ordering put more later-declining pages in the top 50 than the rule baseline did." That's the strongest claim the capstone should aim for.

**What it won't say.**

- That refreshing a page causes recovery. There's no experiment here, and editors pick pages for reasons the data can't see.
- That any signal is a Google ranking factor. These are observational associations on 32 pseudonymized clients.
- That a page is declining just because the starter label says so (the cell below shows why).
- Anything about AI visibility. Only about 6% of pages have even one AI-referred session, and that field counts click-throughs, not citations.
- That a starter-slice result holds on the full warehouse. It has to be re-earned there.

In [4]:
# The starter label can be rebuilt exactly from the current windows, so it
# describes a drop that already happened. It is not a forecast.
prev, last = df["impressions_prev_30d"], df["impressions_last_30d"]
pct = np.where(prev > 0, (last - prev) / prev.where(prev > 0) * 100, np.nan)
rebuilt_down = pd.Series(pct < -20, index=df.index)
print(f"rows where 'down' == (last 30d < prev 30d by more than 20%): "
      f"{(rebuilt_down == down).mean():.1%}")

ai_share = df["ai_sessions_90d"].gt(0).mean()
print(f"pages with at least one AI-referred session: {ai_share:.1%}")

rows where 'down' == (last 30d < prev 30d by more than 20%): 100.0%
pages with at least one AI-referred session: 6.4%


The rebuild matches on 100% of rows. So `trend_direction` and `trend_pct` can never be features, and any honest version of this project needs a target measured after the decision point. That's the first job for the week 2 framing notebook.

## Self-check

- [x] Every section above is filled, markdown thinking and the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere (only pseudonymized aggregates are printed)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`